In [64]:
from Poly import *
from Z import *
import ast

In [73]:
class GF:
    def __init__(self, num : int, q : int | None = None, 
                 poly : list | str | PolyMod |None = None,
                 prime : int | None = None, degree : int | None = None):
        # if p and d are given, use those without checking primality for speed
        if prime is not None and degree is not None:
            p, d  = prime, degree
        else: # check that q = p ^ d is a prime power
            prime_pow = prime_power(q)
            if prime_pow == False:
                raise ValueError('finite field has prime power number of elements')
            else:
                p, d = prime_pow
        #-----------------------------------------------------------------------
        # find the irreducible polynomial to mod by
        #-----------------------------------------------------------------------
        #
        # if f(x) is given, check that it is irreducible of degree d
        if type(poly) == list:
            mod = PolyMod(poly, p)
            # assert mod.deg == d and mod.is_irred(), (
            # 'need irreducible polynomial of degree d to get GF(p^d)')
        elif type(poly) == str:
            assert poly in {'rand', 'random', 'r', 'brute', 'b'}, (
                'options besides conway polynomials are random or brute force'
            )
            if poly in {'rand', 'random', 'r'}:
                import random
                while True:
                    # generate a random polynomial of degree d
                    f = PolyMod(random.randint(p**d + 1, 2 * p**d), p)
                    if f.is_irred(): # check if irreducible, otherwise try again
                        mod = f
                        break
            else: # find the least irred poly of degree d by brute force
                for k in range(p**d + 1, 2 * p**d):
                    if k % p == 0: # not irreducible: divisible by x
                        continue
                    f = PolyMod(k, p)
                    if f.is_irred():
                        mod = f
                        break   
        elif type(poly) == PolyMod: # given directly
            mod = poly
        elif poly == None: # by default, lookup the conway polynomial            
            with open('conway_polynomials.txt', 'r') as file:
                for i, line in enumerate(file):
                    if i == 0: 
                        continue
                    if line[-2] == ';': # got to the end
                        raise ValueError(
                            'The conway polynomial for that field is unknown')
                        
                    data = ast.literal_eval(line.rstrip(',\n'))
                    if data[:2] == [p, d]:
                        mod = PolyMod(data[2], p)
                        break
        else: 
            raise ValueError('need an irreducible polynomial')
        #-----------------------------------------------------------------------
        self.id = num % (p ** d)
        self.repr = PolyMod(self.id, p)
        self.mod = mod
        self.char = p
        self.deg = d
        
    def __add__(self, other) -> object:
        assert self.mod == other.mod, 'need same modulus to add'
        result = (self.repr + other.repr) % self.mod
        return GF(result.id, poly=self.mod, prime=self.char, degree=self.deg)
    
    def __eq__(self, other) -> bool:
        return (self.id == other.id and self.repr == other.repr 
                and self.mod == other.mod)      

    def __mul__(self, other) -> object:
        assert self.mod == other.mod, 'need same modulus to multiply'
        result = (self.repr * other.repr) % self.mod
        return GF(result.id, poly=self.mod, prime=self.char, degree=self.deg)
    
    def __pow__(self, p : int) -> object:
        result = (self.repr ** p) % self.mod
        return GF(result.id, poly=self.mod, prime=self.char, degree=self.deg)
      
    def __repr__(self) -> str:
        return str(self.id)                   


In [ ]:
def conway(p, d):
    with open('conway_polynomials.txt', 'r') as file:
        for i, line in enumerate(file):
            if i == 0: 
                continue
            if line[-2] == ';': # got to the end
                raise ValueError(
                    'The conway polynomial for that field is unknown')
                
            data = ast.literal_eval(line.rstrip(',\n'))
            if data[:2] == [p, d]:
                return PolyMod(data[2], p)

In [102]:
[conway(p, 2) for p in {2, 3, 5, 7, 11, 13, 17, 19, 23, 29}]

[[1, 1, 1],
 [2, 2, 1],
 [2, 4, 1],
 [3, 6, 1],
 [2, 7, 1],
 [2, 12, 1],
 [3, 16, 1],
 [2, 18, 1],
 [5, 21, 1],
 [2, 24, 1]]

In [84]:
a = GF(2, 16)
a.mod, a.repr, a.char, a.deg

([1, 1, 0, 0, 1], [0, 1], 2, 4)

In [85]:
q = 16
p = 13
d = 2
[[GF(k, q, poly=[1, 1, 0, 0, 1]) ** n for k in range(q)] for n in range(q)]

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 [0, 1, 4, 5, 3, 2, 7, 6, 12, 13, 8, 9, 15, 14, 11, 10],
 [0, 1, 8, 15, 12, 10, 1, 1, 10, 15, 15, 12, 8, 10, 8, 12],
 [0, 1, 3, 2, 5, 4, 6, 7, 15, 14, 12, 13, 10, 11, 9, 8],
 [0, 1, 6, 6, 7, 7, 7, 6, 1, 7, 1, 6, 1, 6, 7, 1],
 [0, 1, 12, 10, 15, 8, 1, 1, 8, 10, 10, 15, 12, 8, 12, 15],
 [0, 1, 11, 13, 9, 14, 6, 7, 12, 5, 8, 3, 15, 2, 4, 10],
 [0, 1, 5, 4, 2, 3, 7, 6, 10, 11, 15, 14, 8, 9, 13, 12],
 [0, 1, 10, 12, 8, 15, 1, 1, 15, 12, 12, 8, 10, 15, 10, 8],
 [0, 1, 7, 7, 6, 6, 6, 7, 1, 6, 1, 7, 1, 7, 6, 1],
 [0, 1, 14, 9, 11, 13, 7, 6, 8, 3, 10, 4, 12, 5, 2, 15],
 [0, 1, 15, 8, 10, 12, 1, 1, 12, 8, 8, 10, 15, 12, 15, 10],
 [0, 1, 13, 11, 14, 9, 6, 7, 10, 4, 15, 2, 8, 3, 5, 12],
 [0, 1, 9, 14, 13, 11, 7, 6, 15, 2, 12, 5, 10, 4, 3, 8],
 [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

In [9]:
import ast
s = '[1, 2, 3]'
x = ast.literal_eval(s)
x[0]

1

In [ ]:
p, d = 2, 2
with open('conway_polynomials.txt', 'r') as file:
    for i, line in enumerate(file):
        if i == 0: 
            continue
        if line[-2] == ';': # got to the end
            raise ValueError(
                'The conway polynomial for that field is unknown')
            
        data = ast.literal_eval(line.rstrip(',\n'))
        if data[:2] == [p, d]:
            conway = PolyMod(data[2], p)
            break
conway ** 2

[1, 0, 1, 0, 1]